# Form 4 API: insider buys and sales as JSON

Query every Form 3, 4, 5 and 144 by ticker, insider, transaction code and dollar value. Python and JavaScript, with real responses.

Read it with charts at https://www.quantisnow.com/sec-api/guides/form-4-insider-trading-api. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

A Form 4 is due within two business days of an insider's trade. The SEC publishes it as XML inside a filing, with one row per transaction and codes for what kind of trade it was. The [insider trades endpoint](https://www.quantisnow.com/sec-api/insider-trading) parses that into one record per filing and insider, with the purchases and sales already added up in dollars.

## The fields you'll filter on most

- `transactionCodes`: `P` is an open-market purchase, `S` a sale, `M` an option exercise, `A` a grant, `F` shares withheld for tax, `G` a gift. Only `P` and `S` are trades the insider chose to make at market.
- `purchased.valueUsd` and `sold.valueUsd`: the filing's purchases and sales in dollars, so a filter like `purchased.valueUsd:>250000` works without parsing each row.
- `reportingOwner.cik`: the insider's own CIK. Names are spelled differently from filing to filing; the CIK isn't.
- `reportingOwner.isOfficer`, `isDirector`, `isTenPercentOwner` and `officerTitle`: who they are at the company.
- `rule10b51`: true when the trades were made under a pre-arranged 10b5-1 plan, which makes a sale much less informative.

## Large open-market buys

### Which insiders bought more than $1 million of their own company's stock in the last 30 days?

Large open-market buys are the insider signal most studies find worth watching.

```shell
curl -X POST https://www.quantisnow.com/api/v1/insider-trades \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "formType:4 AND transactionCodes:P AND purchased.valueUsd:>1000000 AND filedAt:[2026-10-25 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"purchased.valueUsd": {"order": "desc"}}]
}'
```

Response (real, first 2 of 136):

```json
{
  "total": {
    "value": 136,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001011438-26-000643:0001307631",
      "accessionNo": "0001011438-26-000643",
      "formType": "4",
      "amendment": false,
      "restatesEarlier": false,
      "filedAt": "2026-11-03T16:35:59-05:00",
      "transactionDate": "2026-11-02",
      "issuer": {
        "cik": "0000789570",
        "name": "MGM Resorts International",
        "ticker": "MGM"
      },
      "reportingOwner": {
        "cik": "0001307631",
        "name": "Meister Keith A.",
        "officerTitle": null,
        "isDirector": true,
        "isOfficer": false,
        "isTenPercentOwner": false
      },
      "transactionCodes": [
        "P"
      ],
      "purchased": {
        "shares": 1745592,
        "valueUsd": 51740827.48
      },
      "sold": {
        "shares": 0,
        "valueUsd": 0
      },
      "sharesOwnedAfter": null,
      "rule10b51": false,
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/789570/000101143826000643/0001011438-26-000643-index.htm"
    },
    {
      "id": "0001104659-26-121503:0001820515",
      "accessionNo": "0001104659-26-121503",
      "formType": "4/A",
      "amendment": true,
      "restatesEarlier": false,
      "filedAt": "2026-11-02T17:47:09-05:00",
      "transactionDate": "2026-10-29",
      "issuer": {
        "cik": "0000084839",
        "name": "Rollins Inc",
        "ticker": "ROL"
      },
      "reportingOwner": {
        "cik": "0001820515",
        "name": "R. Randall Rollins Voting Trust U/A dated August 25, 1994",
        "officerTitle": null,
        "isDirector": false,
        "isOfficer": false,
        "isTenPercentOwner": true
      },
      "transactionCodes": [
        "P"
      ],
      "purchased": {
        "shares": 1444402,
        "valueUsd": 49956665.33
      },
      "sold": {
        "shares": 0,
        "valueUsd": 0
      },
      "sharesOwnedAfter": null,
      "rule10b51": false,
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/84839/000110465926121503/0001104659-26-121503-index.htm"
    }
  ]
}
```

## One person's sales, by CIK

### How much Snowflake stock has Frank Slootman sold in the last six months?

Follow one person across every filing by their CIK, not their name, which is spelled differently from form to form.

```shell
curl -X POST https://www.quantisnow.com/api/v1/insider-trades \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "reportingOwner.cik:0001402348 AND sold.valueUsd:>0 AND filedAt:[2026-05-28 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"filedAt": {"order": "desc"}}]
}'
```

Response (real, first 2 of 13):

```json
{
  "total": {
    "value": 13,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001402348-26-000021:0001402348",
      "accessionNo": "0001402348-26-000021",
      "formType": "4",
      "amendment": false,
      "restatesEarlier": false,
      "filedAt": "2026-11-20T19:04:09-05:00",
      "transactionDate": "2026-11-19",
      "issuer": {
        "cik": "0001640147",
        "name": "Snowflake Inc.",
        "ticker": "SNOW"
      },
      "reportingOwner": {
        "cik": "0001402348",
        "name": "Slootman Frank",
        "officerTitle": null,
        "isDirector": true,
        "isOfficer": false,
        "isTenPercentOwner": false
      },
      "transactionCodes": [
        "M",
        "S"
      ],
      "purchased": {
        "shares": 0,
        "valueUsd": 0
      },
      "sold": {
        "shares": 300000,
        "valueUsd": 84365360.07
      },
      "sharesOwnedAfter": 19024,
      "rule10b51": true,
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1640147/000140234826000021/0001402348-26-000021-index.htm"
    },
    {
      "id": "0001628280-26-063114:0001402348",
      "accessionNo": "0001628280-26-063114",
      "formType": "4",
      "amendment": false,
      "restatesEarlier": false,
      "filedAt": "2026-09-22T19:18:29-04:00",
      "transactionDate": "2026-09-21",
      "issuer": {
        "cik": "0001640147",
        "name": "Snowflake Inc.",
        "ticker": "SNOW"
      },
      "reportingOwner": {
        "cik": "0001402348",
        "name": "Slootman Frank",
        "officerTitle": null,
        "isDirector": true,
        "isOfficer": false,
        "isTenPercentOwner": false
      },
      "transactionCodes": [
        "M",
        "S"
      ],
      "purchased": {
        "shares": 0,
        "valueUsd": 0
      },
      "sold": {
        "shares": 300000,
        "valueUsd": 100345541.19
      },
      "sharesOwnedAfter": 19024,
      "rule10b51": true,
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1640147/000162828026063114/0001628280-26-063114-index.htm"
    }
  ]
}
```

## In Python

In [ ]:
import os, requests

BASE = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]

def search(endpoint, query, size=50, sort=None, start=0):
    body = {"query": query, "size": size, "from": start}
    if sort:
        body["sort"] = sort
    res = requests.post(f"{BASE}/{endpoint}", json=body,
                        headers={"Authorization": f"Bearer {KEY}"}, timeout=30)
    res.raise_for_status()
    return res.json()

# Officers buying their own stock in the last two weeks, $100k or more
page = search(
    "insider-trades",
    "formType:4 AND transactionCodes:P AND reportingOwner.isOfficer:true "
    "AND purchased.valueUsd:>=100000 AND filedAt:[2026-11-09 TO 2026-11-23]",
    sort="purchased.valueUsd:desc",
)
for r in page["data"]:
    o = r["reportingOwner"]
    print(r["issuer"]["ticker"], o["name"], o["officerTitle"], round(r["purchased"]["valueUsd"]))

## In JavaScript

```javascript
const BASE = "https://www.quantisnow.com/api/v1";

async function search(endpoint, query, { size = 50, sort, from = 0 } = {}) {
  const res = await fetch(BASE + "/" + endpoint, {
    method: "POST",
    headers: {
      Authorization: "Bearer " + process.env.QUANTISNOW_API_KEY,
      "Content-Type": "application/json",
    },
    body: JSON.stringify({ query, size, from, sort }),
  });
  if (!res.ok) throw new Error((await res.json()).error.message);
  return res.json();
}

// Sales over $5M that were not under a 10b5-1 plan
const page = await search(
  "insider-trades",
  "formType:4 AND transactionCodes:S AND sold.valueUsd:>5000000 AND rule10b51:false",
  { sort: "sold.valueUsd:desc", size: 20 },
);
for (const r of page.data) {
  console.log(r.issuer.ticker, r.reportingOwner.name, Math.round(r.sold.valueUsd));
}
```

To group purchases into clusters of several insiders buying the same stock, see [insider buying clusters](https://www.quantisnow.com/sec-api/guides/insider-buying-clusters). For selling that hasn't happened yet, see [Form 144](https://www.quantisnow.com/sec-api/guides/form-144-insider-selling).